In [ ]:
# imports

import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import numpy as np
from scipy.signal import savgol_filter
import scipy.stats as stats


# Configuração do diretório
data_dir = r"c:\Users\gabri\Documents\PROJETOS\POLI\Lab Física\attachments"
sem_file = os.path.join(data_dir, "sem.csv")

# Carregar dados de ruído (fundo)
df_sem = pd.read_csv(sem_file, sep=";", decimal=",")

In [ ]:
# Dados

# Frequências
frequencias = {
    "Vermelho": 4.875,
    "Amarelo": 5.187,  # Média das linhas do dubleto (5.196 e 5.178)
    "Verde": 5.490,
    "Azul": 6.879,
    "Violeta": 7.409,
    "Ultra-Violeta": 8.213,
}

In [ ]:
# Ordem e cores

# Ordem: Amarelo, Verde, Azul, Violeta, Ultra-Violeta, Vermelho
colors_info = [
    {
        "name": "Amarelo",
        "prefix": "amar",
        "letter": "(a)",
        "xlim": (-1.5, 0.5),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Verde",
        "prefix": "verde",
        "letter": "(b)",
        "xlim": (-2.5, 0.5),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Azul",
        "prefix": "azul",
        "letter": "(c)",
        "xlim": (-3.0, 0.5),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Violeta",
        "prefix": "vio",
        "letter": "(d)",
        "xlim": (-3.5, 0.5),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Ultra-Violeta",
        "prefix": "UV",
        "letter": "(e)",
        "xlim": (-4.0, 0.5),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Vermelho",
        "prefix": "ver",
        "letter": "(f)",
        "xlim": (-1.5, 1.0),
        "ylim": (-1.5, 1.5),
    },
]

# Intensidades cores
intensity_colors = {
    100: "tab:green",
    80: "#f9d71c",
    60: "tab:blue",
    40: "tab:red",
    20: "tab:gray",
}

intensities = [100, 80, 60, 40, 20]

# Estilo global dos gráficos
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 12
plt.rcParams["axes.axisbelow"] = True

In [ ]:
# Gráficos das medições

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes_flat = axes.flatten()

for i, info in enumerate(colors_info):
    ax = axes_flat[i]

    # Ordenar intensidades para legenda (descendente conforme imagem)
    intensities_to_plot = [100, 80, 60, 40, 20]

    for intensity in intensities_to_plot:
        file_path = os.path.join(data_dir, f"{info['prefix']}-{intensity}.csv")

        if os.path.exists(file_path):
            df = pd.read_csv(file_path, sep=";", decimal=",")

            corrente = df["Corrente [A]"]
            if df_sem is not None and len(df) == len(df_sem):
                corrente = corrente - df_sem["Corrente [A]"]

            corrente_plot = corrente * 1e10
            tensao = df["Tensao [V]"]

            ax.plot(
                tensao,
                corrente_plot,
                label=f"{intensity}%",
                color=intensity_colors.get(intensity, "black"),
                linewidth=1.5,
            )

    # Configurações estéticas por subplot
    ax.set_xlabel("Potencial Aplicado (V)", fontsize=12)
    ax.set_ylabel(r"Corrente ($x10^{-10}$ A)", fontsize=12)

    # Letra (a, b, c...) no topo direito
    ax.text(
        0.95,
        0.95,
        info["letter"],
        transform=ax.transAxes,
        fontsize=16,
        va="top",
        ha="right",
    )

    # Nome da cor no canto inferior direito
    ax.text(
        0.98,
        0.02,
        info["name"],
        transform=ax.transAxes,
        fontsize=14,
        va="bottom",
        ha="right",
    )

    # Grid e ticks
    ax.grid(True, which="major", linestyle="--", color="gray", alpha=0.6)
    ax.minorticks_on()
    ax.grid(True, which="minor", linestyle=":", color="gray", alpha=0.3)

    # Legenda
    ax.legend(
        loc="upper left", fontsize=10, frameon=True, edgecolor="black", fancybox=False
    )

    # Limites de escala
    if info["xlim"]:
        ax.set_xlim(info["xlim"])
    if info["ylim"]:
        ax.set_ylim(info["ylim"])

    ax.tick_params(axis="both", which="major", labelsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Cálculo de V0 pela segunda derivada


def calculate_v0_derivative_method(df_signal, df_noise):
    """
    Calcula V0 baseado no critério da segunda derivada.
    """
    # 1. Limpeza por Subtração
    if df_noise is not None and len(df_signal) == len(df_noise):
        corrente_limpa = df_signal["Corrente [A]"] - df_noise["Corrente [A]"]
    else:
        corrente_limpa = df_signal["Corrente [A]"]

    voltagem = df_signal["Tensao [V]"].values
    corrente = corrente_limpa.values

    # Ordenar por tensão (do mais negativo para o positivo) se necessário
    sort_idx = np.argsort(voltagem)
    voltagem = voltagem[sort_idx]
    corrente = corrente[sort_idx]

    # 2. Suavização e Derivada
    # Usamos Savitzky-Golay para suavizar e calcular a 2ª derivada simultaneamente
    # window_length deve ser ímpar e ajustado à densidade de pontos.
    # polyorder 3 é bom para capturar curvaturas.
    # deriv=2 calcula diretamente a segunda derivada.
    d2I_dV2 = savgol_filter(corrente, window_length=21, polyorder=3, deriv=2)

    # 3. Definição da Faixa de Ruído (Baseline)
    # Selecionar região "profunda" onde certamente não há fotocorrente (ex: -9V a -4V)
    # Ajuste este intervalo conforme seus dados reais. Olhando o CSV, começa em -10.
    mask_baseline = (voltagem > -9.0) & (voltagem < -4.0)

    if np.sum(mask_baseline) < 5:
        return np.nan  # Dados insuficientes na baseline

    baseline_data = d2I_dV2[mask_baseline]

    # Média e Desvio Padrão do ruído na segunda derivada
    mu_noise = np.mean(baseline_data)
    sigma_noise = np.std(baseline_data)

    # 4. Critério de Decisão (Varredura da esquerda para a direita)
    # Procuramos onde o sinal sai de mu +/- 1*sigma
    threshold_upper = mu_noise + 1.0 * sigma_noise

    # Começamos a busca a partir do fim da baseline
    start_search_v = -4.0
    mask_search = voltagem > start_search_v

    v_search = voltagem[mask_search]
    signal_search = d2I_dV2[mask_search]

    consecutive_points = 3  # Exige que n pontos seguidos passem o limiar para confirmar
    v0_encontrado = np.nan

    for i in range(len(signal_search) - consecutive_points):
        # Verifica se n pontos seguidos excedem o limiar (indicando curvatura real)
        if np.all(signal_search[i : i + consecutive_points] > threshold_upper):
            v0_encontrado = v_search[i]
            break

    return v0_encontrado


# Estrutura para salvar resultados consolidados
resultados_finais = []

print(f"{'Cor':<15} {'Intensidade':<12} {'V0 (V)':<10}")
print("-" * 40)

for info in colors_info:
    v0_values = []

    for intensity in intensities:  # [100, 80, 60, 40, 20]
        file_path = os.path.join(data_dir, f"{info['prefix']}-{intensity}.csv")

        if os.path.exists(file_path):
            df_sinal = pd.read_csv(file_path, sep=";", decimal=",")

            v0 = calculate_v0_derivative_method(df_sinal, df_sem)

            if not np.isnan(v0):
                v0_values.append(v0)
                print(f"{info['name']:<15} {intensity:<12} {v0:.4f}")
            else:
                print(f"{info['name']:<15} {intensity:<12} Falha")

    if v0_values:
        v0_array = np.array(v0_values)
        media_v0 = np.mean(v0_array)
        # Erro padrão da média = desvio padrão da amostra / raiz(n)
        erro_v0 = np.std(v0_array, ddof=1) / np.sqrt(len(v0_array))

        resultados_finais.append(
            {
                "Cor": info["name"],
                "V0_medio": media_v0,
                "Incerteza": erro_v0,
                "N_medidas": len(v0_array),
            }
        )

print("\n=== RESULTADOS CONSOLIDADOS PARA O GRÁFICO DE PLANCK ===")
df_resultados = pd.DataFrame(resultados_finais)
print(df_resultados)

In [ ]:
# Cálculo de V0 pela primeira derivada


def calculate_v0_tangent_method(df_signal, df_noise):
    """
    Calcula V0 usando o Método da Tangente (Extrapolação Linear).
    1. Limpa o sinal e suaviza (janela maior).
    2. Encontra o ponto de máxima inclinação (primeira derivada).
    3. Ajusta uma reta nessa região de subida.
    4. Encontra a interseção dessa reta com a linha de base (média do ruído).
    """
    # 1. Limpeza e Extração
    if df_noise is not None and len(df_signal) == len(df_noise):
        corrente_limpa = df_signal["Corrente [A]"] - df_noise["Corrente [A]"]
    else:
        corrente_limpa = df_signal["Corrente [A]"]

    voltagem = df_signal["Tensao [V]"].values
    corrente = corrente_limpa.values

    # Ordenar
    sort_idx = np.argsort(voltagem)
    voltagem = voltagem[sort_idx]
    corrente = corrente[sort_idx]

    # 2. Suavização (Item C: Janela aumentada para 51)
    # Se houver poucos pontos, ajusta a janela para evitar erro
    window = 51 if len(corrente) > 55 else 11
    corrente_smooth = savgol_filter(corrente, window_length=window, polyorder=3)

    # 3. Calcular Primeira Derivada para achar a região de subida
    # Restringimos a busca da inclinação máxima a uma região onde esperamos o fenômeno
    # (Ex: entre -4V e +1V) para evitar pegar ruidos nas pontas
    mask_slope_search = (voltagem > -3.0) & (voltagem < 1.0)

    if np.sum(mask_slope_search) == 0:
        return np.nan

    dI_dV = np.gradient(corrente_smooth, voltagem)

    # Encontra o índice da inclinação máxima DENTRO da região de interesse
    indices_search = np.where(mask_slope_search)[0]
    idx_relative_max = np.argmax(dI_dV[indices_search])
    idx_max_slope = indices_search[idx_relative_max]

    # 4. Ajuste Linear (Tangente)
    # Pegamos alguns pontos ao redor do máximo de inclinação para traçar a reta
    n_points_fit = 10
    start_fit = max(0, idx_max_slope - n_points_fit)
    end_fit = min(len(voltagem), idx_max_slope + n_points_fit)

    V_fit = voltagem[start_fit:end_fit]
    I_fit = corrente_smooth[start_fit:end_fit]

    # y = mx + c
    slope, intercept = np.polyfit(V_fit, I_fit, 1)

    # 5. Calcular Linha de Base (Baseline)
    # Região onde definitivamente não há sinal (Item A/B: referência de zero)
    mask_baseline = (voltagem > -9.0) & (voltagem < -4.0)
    if np.sum(mask_baseline) > 5:
        baseline_level = np.mean(corrente_smooth[mask_baseline])
    else:
        baseline_level = 0.0  # Fallback se não tiver dados negativos o suficiente

    # 6. Encontrar Interseção: mx + c = baseline
    # x = (baseline - c) / m
    v0 = (baseline_level - intercept) / slope

    return v0


# --- Execução e Tabela ---

resultados_finais = []

print(f"{'Cor':<15} {'Intensidade':<12} {'V0 calc. (V)':<15}")
print("=" * 45)

for info in colors_info:
    v0_values = []

    for intensity in intensities:  # [100, 80, 60, 40, 20]
        file_path = os.path.join(data_dir, f"{info['prefix']}-{intensity}.csv")

        if os.path.exists(file_path):
            df_sinal = pd.read_csv(file_path, sep=";", decimal=",")

            # Usando o novo método da tangente
            v0 = calculate_v0_tangent_method(df_sinal, df_sem)

            # Filtro básico de sanidade (V0 deve estar entre -4V e 0V para este exp)
            if not np.isnan(v0) and -5.0 < v0 < 0.5:
                v0_values.append(v0)
                print(f"{info['name']:<15} {intensity:<12} {v0:.4f}")
            else:
                print(f"{info['name']:<15} {intensity:<12} {v0:.4f} (Desc.)")

    if v0_values:
        v0_array = np.array(v0_values)
        media_v0 = np.mean(v0_array)
        erro_v0 = np.std(v0_array, ddof=1) / np.sqrt(len(v0_array))

        resultados_finais.append(
            {
                "Cor": info["name"],
                "V0_medio": media_v0,
                "Incerteza": erro_v0,
                "N_medidas": len(v0_array),
            }
        )

print("\n=== TABELA FINAL PARA GRÁFICO DE PLANCK ===")
df_resultados = pd.DataFrame(resultados_finais)
# Exibindo colunas formatadas
print(df_resultados[["Cor", "V0_medio", "Incerteza", "N_medidas"]])

In [ ]:
# Curva de V0 x f


def plot_planck_graph(df_res, title_suffix=""):
    # Preparar dados: converter V0 para valor absoluto (magnitude)
    df_plot = df_res.copy()
    df_plot["f"] = df_plot["Cor"].map(frequencias)
    df_plot["V0_abs"] = np.abs(df_plot["V0_medio"])  # Converter para módulo

    # Remover falhas
    df_plot = df_plot.dropna(subset=["f", "V0_abs"])
    x = df_plot["f"].values
    y = df_plot["V0_abs"].values
    y_err = df_plot["Incerteza"].values

    # Ajuste Linear
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    line = slope * x + intercept

    # Cálculo da Banda de Confiança (68% -> 1 sigma)
    # Aproximação simples para o erro da predição
    n = len(x)
    x_grid = np.linspace(4.5, 8.5, 100)
    y_grid = slope * x_grid + intercept

    # Erro residual
    resid = y - (slope * x + intercept)
    s_err = np.sqrt(np.sum(resid**2) / (n - 2))

    conf_band = s_err * np.sqrt(
        1 / n + (x_grid - np.mean(x)) ** 2 / np.sum((x - np.mean(x)) ** 2)
    )

    # Plotagem
    fig, ax = plt.subplots(figsize=(8, 8))

    # Pontos e Erro
    ax.errorbar(x, y, yerr=y_err, fmt="ok", capsize=3, label=r"$V_0$")

    # Linha e Banda
    ax.plot(x_grid, y_grid, "r-", linewidth=2, label=f"Linear Fit {title_suffix}")
    ax.fill_between(
        x_grid,
        y_grid - conf_band,
        y_grid + conf_band,
        color="red",
        alpha=0.3,
        label="68% Confidence Band",
    )

    # Estética
    ax.set_xlabel(r"Frequência da Radiação ($x10^{14} Hz$)", fontsize=14)
    ax.set_ylabel(r"Potencial de Corte $V_0$ (V)", fontsize=14)
    ax.set_xlim(4.5, 8)
    ax.set_ylim(0.5, 2)

    # Tabela de Resultados (Ajuste conforme imagem)
    stats_text = (
        f"Equation: y = a + b*x\n"
        f"Intercept (a): {intercept:.5f}\n"
        f"Slope (b): {slope:.5e}\n"
        f"R-Square: {r_value**2:.5f}\n"
        f"h_calc: {slope * 1.602e-19 * 1e-14:.2e} J.s"
    )
    ax.text(
        0.55,
        0.2,
        stats_text,
        transform=ax.transAxes,
        fontsize=10,
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray"),
    )

    ax.legend(loc="upper left", frameon=True, edgecolor="black")
    ax.grid(True, linestyle=":", alpha=0.6)
    plt.show()


# Gerar para ambos os métodos (supondo que você salvou os dataframes)
print("Gráfico: Método da Tangente")
plot_planck_graph(df_resultados, "(Met. Tangente)")